In [21]:
from collections import Counter
from IPython.display import display
import pandas as pd
import glob
import os
import warnings


In [22]:
DATA_PATH = r'input'

In [23]:
with open('events.txt') as f:
    event_names = tuple(line.strip() for line in f if line.strip())

event_names[:5]

('ConsoleLogin', 'CreateAccount', 'ListBuckets')

In [24]:

def load_file(path):
    df = pd.read_csv(path)
    all_nan = [c for c in df.columns[:-1] if df[c].isna().all()]
    if all_nan:
        raise ValueError(f"Columns entirely NaN in {path}: {all_nan}")
    slash_cols = [c for c in df.columns if c not in ('userAgent', 'userIdentity.arn')]
    bad_slash = df[slash_cols].astype(str).apply(lambda col: col.str.contains('/', na=False)).any(axis=1)
    if bad_slash.any():
        warnings.warn(f"Unexpected '/' in non-userAgent fields in {path}: rows {df.index[bad_slash].tolist()}")
    bad_source = ~df['eventSource'].astype(str).str.endswith('.amazonaws.com')
    if bad_source.any():
        warnings.warn(f"Invalid eventSource values in {path}: {df.loc[bad_source, 'eventSource'].unique().tolist()}")
    return df

files = glob.glob(os.path.join(DATA_PATH, '*.csv'))
signals_raw = pd.concat((load_file(f) for f in files), ignore_index=True)
signals_raw.head()

,eventID,eventTime,sourceIPAddress,eventSource,eventName,userAgent,userIdentity.arn,tactic1,tactic2,rank,type
0,22741,2018-09-26T18:09:00.000Z,198.18.161.170,billingconsole.amazonaws.com,AWSPaymentInstrumentGateway.Delete,Null,arn:aws:iam::123456789012:root,Defense Evasion,-,5,pyod
1,22742,2018-09-28T13:39:07.000Z,198.18.161.170,signin.amazonaws.com,ConsoleLogin,AWSBFF BFFiOS/1.19.6 (1015) Mobile iOS/11.4.1 ...,arn:aws:iam::123456789012:root,Persistence,Initial Access,3,pyod
2,22743,2018-09-27T15:28:29.000Z,198.18.161.170,signin.amazonaws.com,ConsoleLogin,AWSBFF BFFiOS/1.19.6 (1015) Mobile iOS/11.4.1 ...,arn:aws:iam::123456789012:root,Persistence,Initial Access,4,pyod
3,22744,2018-09-30T17:30:21.000Z,223.0.113.35,signin.amazonaws.com,ConsoleLogin,AWSBFF BFFiOS/1.19.6 (1015) Mobile iOS/11.4.1 ...,arn:aws:iam::123456789012:root,Persistence,Initial Access,3,pyod
4,22745,2018-09-29T13:58:43.000Z,198.18.161.170,signin.amazonaws.com,ConsoleLogin,AWSBFF BFFiOS/1.19.6 (1015) Mobile iOS/11.4.1 ...,arn:aws:iam::123456789012:root,Persistence,Initial Access,3,pyod


In [ ]:
# signal processing 

def extract_user(arn):
    if arn.startswith('arn:aws:sts') and '/assumed-role/' in arn:
        return arn.split('/')[-2]
    return arn.split('/')[-1] if '/' in arn else arn.split(':')[-1]

# corroborated: events detected by both pyod and kmeans
types_per_event = signals_raw.groupby('eventID')['type'].nunique()
duplicate_ids = types_per_event[types_per_event > 1].index
corroborated = (
    signals_raw[signals_raw['eventID'].isin(duplicate_ids)]
    .drop(columns=['outlier_kmeans_score', 'rank'], errors='ignore')
    .assign(type='Signal Fusion: Model Convergence')
    .drop_duplicates()
    .sort_values('eventID')
    .reset_index(drop=True)
)

# spec_based: events matching specification rules
spec_based = (
    signals_raw[signals_raw['eventName'].isin(event_names)]
    .drop(columns=['rank', 'outlier_kmeans_score'], errors='ignore')
    .assign(type='Specification Based Detection')
    .drop_duplicates()
)

# signal_fusion: spec_based events also caught by ML
ml_ids = signals_raw[signals_raw['type'].isin(['pyod', 'kmeans'])]['eventID']
signal_fusion = (
    spec_based[spec_based['eventID'].isin(ml_ids)]
    .drop(columns=['rank', 'outlier_kmeans_score'], errors='ignore')
    .copy()
    .drop_duplicates()
    .assign(**{'type': 'Signal Fusion: ML and Conventional Alert'})
)

# all_detections tuple
all_detections = tuple(
    (row['eventID'], row['type'])
    for df in [corroborated, spec_based, signal_fusion]
    for _, row in df[['eventID', 'type']].drop_duplicates().iterrows()
)

# signals_processed
signals_processed = pd.concat([
    signals_raw.drop(columns=['outlier_kmeans_score'], errors='ignore'),
    corroborated,
    spec_based,
    signal_fusion
], ignore_index=True).drop_duplicates()

signals_processed['user'] = signals_processed['userIdentity.arn'].apply(extract_user)




In [31]:
type_rename = {
    'pyod':                                      'ML Anomaly Detection: PYOD',
    'kmeans':                                    'ML Anomaly Detection: K-means',
    'Signal Fusion: Model Convergence':          'Signal Fusion: Model Convergence',
    'Signal Fusion: ML and Conventional Alert':  'Signal Fusion: ML and Conventional Alert',
    'Specification Based Detection':             'Specification Based Detection',
}

signals_processed['type'] = signals_processed['type'].replace(type_rename)
signals_processed['type'].value_counts()

# per-ARN breakdown with totals
pivot = signals_processed.pivot_table(
    index='userIdentity.arn',
    columns='type',
    aggfunc='size',
    fill_value=0
)

pd.concat([pivot, pivot.sum().rename('Total').to_frame().T])

type,ML Anomaly Detection: K-means,ML Anomaly Detection: PYOD,Signal Fusion: ML and Conventional Alert,Signal Fusion: Model Convergence,Specification Based Detection
arn:aws:iam::123456789012:root,127,127,35,127,35
arn:aws:sts::123456789012:assumed-role/flowlogsRole/vpc-flow-logging+123456789012,4,4,0,4,0
Total,131,131,35,131,35


In [32]:
# scoring
import numpy as np
import ipaddress

EXCLUDE_PRIVATE_IPS = True

type_weights = {
    'ML Anomaly Detection: PYOD': 1,
    'ML Anomaly Detection: K-means:': 1,
    'specification based detection': 2,
    'Signal Fusion: Model Convergence': 3,
    'Signal Fusion: ML and Conventional Alert': 4,
}

def is_private_ip(addr):
    try:
        return ipaddress.ip_address(addr).is_private
    except ValueError:
        return False

def arn_score(group):
    # pyod rows weighted by log1p(rank); all other types count as 1 each
    diversity = sum(
        type_weights.get(t, 1) * np.log1p(sub['_rank_w'].sum())
        for t, sub in group.groupby('type')
    )
    return diversity * np.log1p(len(group))

def compute_scores(df, group_by):
    df = df.copy()
    df['_rank_w'] = 1.0
    pyod_mask = df['type'] == 'pyod'
    if 'rank' in df.columns and pyod_mask.any():
        df.loc[pyod_mask, '_rank_w'] = np.log1p(
            pd.to_numeric(df.loc[pyod_mask, 'rank'], errors='coerce').fillna(1).clip(lower=1)
        )
    aw = df.groupby(group_by)[['type', '_rank_w']].apply(arn_score).rename('atomic weight').round(1)
    all_t = set(df['type'].unique())
    hat = df.groupby(group_by)['type'].apply(set).apply(lambda s: s >= all_t)
    an = (
        df[df['type'].isin(fusion_types)]
        .groupby(group_by).size()
        .reindex(aw.index, fill_value=0)
        .rename('atomic number')
    )
    bw = aw * hat.map({True: FULL_COVERAGE_BOOST, False: 1.0}) * (1 + FUSION_SCALE * np.log1p(an))
    sc = bw.rename('score').round(1)
    am = df.groupby(group_by).size().rename('atomic mass')
    return pd.concat([aw, an, am, sc], axis=1).sort_values('score', ascending=False)

fusion_types = {'Signal Fusion: ML and Conventional Alert', 'Signal Fusion: Model Convergence'}
FULL_COVERAGE_BOOST = 2
FUSION_SCALE = 0.1

sp = signals_processed
if EXCLUDE_PRIVATE_IPS:
    sp = sp[~sp['sourceIPAddress'].apply(is_private_ip)]

scores_by_arn  = compute_scores(sp, 'userIdentity.arn')
scores_by_user = compute_scores(sp, 'user')
scores_by_ip   = compute_scores(sp, 'sourceIPAddress')

In [33]:
import ipywidgets as widgets
from IPython.display import clear_output

results = {
    'userIdentity.arn': scores_by_arn,
    'user': scores_by_user,
    'sourceIPAddress': scores_by_ip,
}
dropdown = widgets.Dropdown(options=list(results), value='userIdentity.arn', description='View by:')

def show_scores(change=None):
    clear_output(wait=True)
    display(dropdown)
    with pd.option_context('display.max_colwidth', None):
        display(results[dropdown.value])

dropdown.observe(show_scores, names='value')
show_scores()

Dropdown(description='View by:', options=('userIdentity.arn', 'user', 'sourceIPAddress'), value='userIdentity.…

,atomic weight,atomic number,atomic mass,score
userIdentity.arn,,,,
arn:aws:iam::123456789012:root,62.3,14,41,158.3
arn:aws:sts::123456789012:assumed-role/flowlogsRole/vpc-flow-logging+123456789012,20.6,4,12,23.9


In [ ]:
import ipywidgets as widgets
from IPython.display import HTML, clear_output

field_dd = widgets.Dropdown(options=["user", "userIdentity.arn"], value="user", description="Group by:")

def render(change=None):
    clear_output(wait=True)
    display(field_dd)
    field = field_dd.value
    score_table = scores_by_user if field == "user" else scores_by_arn
    for val in sorted(sp[field].dropna().unique()):
        subset = sp[sp[field] == val]
        pivot = subset.pivot_table(
            index="eventName",
            columns="type",
            aggfunc="size",
            fill_value=0
        )
        row = score_table.loc[val] if val in score_table.index else None
        meta = (
            f' &nbsp;<span style="font-weight:normal;font-size:0.85em;color:#555">atomic weight {row["atomic weight"]} &nbsp;&middot;&nbsp; score {row["score"]}</span>'
            if row is not None else ""
        )
        display(HTML(f'<h3 style="margin-top:1.5em;border-bottom:1px solid #ccc;padding-bottom:4px">{val}{meta}</h3>'))
        with pd.option_context("display.max_colwidth", None):
            display(pivot)

field_dd.observe(render, names="value")
render()


Dropdown(description='Group by:', index=1, options=('user', 'userIdentity.arn'), value='userIdentity.arn')

type,ML Anomaly Detection: K-means,ML Anomaly Detection: PYOD,Signal Fusion: ML and Conventional Alert,Signal Fusion: Model Convergence,Specification Based Detection
eventName,,,,,
ConsoleLogin,1,1,1,1,1
DescribeAvailabilityZones,4,4,0,4,0
DescribeDBInstances,1,1,0,1,0
DescribeLoadBalancers,7,7,0,7,0


type,ML Anomaly Detection: K-means,ML Anomaly Detection: PYOD,Signal Fusion: Model Convergence
eventName,,,
CreateLogStream,4,4,4
